# Notebook to inspect and save lapresse database

In [1]:
import sqlite3
import pandas as pd
import os

In [ ]:
conn = sqlite3.connect('../../data/lapresse.db')
cursor = conn.cursor()

In [3]:
# Fetch all table names
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = [row[0] for row in cursor.fetchall()]

# Loop through tables and print their schema (column names & types)
print(f"\n📊 Database contains {len(tables)} tables:\n")

for table in tables:
    print(f"🔹 Table: {table}")
    cursor.execute(f"PRAGMA table_info({table});")
    columns = cursor.fetchall()
    
    for col in columns:
        col_name, col_type = col[1], col[2]
        print(f"    - {col_name} ({col_type})")
    print()


📊 Database contains 5 tables:

🔹 Table: entry
    - id (INTEGER)
    - url (VARCHAR(255))
    - created (DATETIME)
    - checked (DATETIME)

🔹 Table: diff
    - id (INTEGER)
    - old_id (INTEGER)
    - new_id (INTEGER)
    - created (DATETIME)
    - tweeted (DATETIME)
    - blogged (DATETIME)

🔹 Table: feed
    - url (VARCHAR(255))
    - name (VARCHAR(255))
    - created (DATETIME)

🔹 Table: feedentry
    - id (INTEGER)
    - feed_id (VARCHAR(255))
    - entry_id (INTEGER)
    - created (DATETIME)

🔹 Table: entryversion
    - index (INTEGER)
    - id (INTEGER)
    - title (TEXT)
    - url (TEXT)
    - summary (TEXT)
    - created (TEXT)
    - archive_url (TEXT)
    - entry_id (INTEGER)
    - num_versions (INTEGER)
    - version (REAL)



In [4]:
df = pd.read_sql_query("SELECT * FROM entryversion", conn)

# 1️⃣ Basic stats - Considering all versions
print("🔎 Basic statistics - ALL versions:")

total_versions = len(df)
unique_articles = df['entry_id'].nunique()

df['word_count'] = df['summary'].apply(lambda x: len(str(x).split()))

print(f"Total article versions: {total_versions}")
print(f"Unique articles (entry_id): {unique_articles}")
print(f"Average word count per summary: {df['word_count'].mean():.2f}")
print(f"Min word count: {df['word_count'].min()}")
print(f"Max word count: {df['word_count'].max()}")

print("\n")

# 2️⃣ Basic stats - Considering only the newest version per article
print("🔎 Basic statistics - NEWEST version per article:")

# Sort by version number (higher version = newer) and keep only the latest per entry_id
df_latest = df.sort_values(['entry_id', 'version'], ascending=[True, False]).drop_duplicates(subset=['entry_id'], keep='first')

total_latest = len(df_latest)
print(f"Total articles (only newest version): {total_latest}")
print(f"Average word count per summary (newest versions only): {df_latest['word_count'].mean():.2f}")
print(f"Min word count (newest versions only): {df_latest['word_count'].min()}")
print(f"Max word count (newest versions only): {df_latest['word_count'].max()}")

🔎 Basic statistics - ALL versions:
Total article versions: 73447
Unique articles (entry_id): 40978
Average word count per summary: 463.98
Min word count: 0
Max word count: 5167


🔎 Basic statistics - NEWEST version per article:
Total articles (only newest version): 40978
Average word count per summary (newest versions only): 510.53
Min word count (newest versions only): 0
Max word count (newest versions only): 5106


In [5]:
df_top6000 = df_latest.sort_values('word_count', ascending=False).head(6000)

# Print basic statistics for the final dataset
print("🔎 Basic statistics - Top 6000 longest articles (newest versions only):")

total_articles = len(df_top6000)
average_word_count = df_top6000['word_count'].mean()
min_word_count = df_top6000['word_count'].min()
max_word_count = df_top6000['word_count'].max()

print(f"Average word count: {average_word_count:.2f}")
print(f"Min word count: {min_word_count}")
print(f"Max word count: {max_word_count}")

🔎 Basic statistics - Top 6000 longest articles (newest versions only):
Average word count: 1082.33
Min word count: 815
Max word count: 5106


In [6]:
# Article with the most words (first row in df_top5000 after sorting)
longest_article = df_top6000.iloc[0]

# Print its summary
print(f"\n🔝 Article with the most words (word count = {longest_article['word_count']}):\n")
print(longest_article['summary'])


🔝 Article with the most words (word count = 5106):

&#13; &#13; &#13; &#13; &#13; 	 		<p>Au cours des dernières semaines, une vingtaine de danseurs ont porté plainte à l'Union des artistes (UDA) contre le chorégraphe vedette Steve Bolton, qui travaille à la télé dans les émissions Les dieux de la danse et La voix ainsi que dans des comédies musicales à succès comme Mary Poppins. « Extrêmement inquiète » de la nature des allégations reçues, qui vont de la violence physique et psychologique aux conditions de travail intenables, l'UDA compte exercer une « vigie accrue » à son endroit. La Presse a obtenu une copie de 11 plaintes et a parlé à 18 danseurs qui affirment avoir souffert sous la coupe de Steve Bolton... ainsi qu'à d'autres, qui le défendent.</p> &#13; 	&#13; &#13; &#13; &#13; &#13; 	Une vingtaine de plaintes de danseurs&#13; &#13; 	&#13; 	<p>Violence physique et verbale lors des répétitions. Crises de colère. Attitude générale d'abus de pouvoir. Refus de donner des pauses aux d

In [ ]:
import os
import pandas as pd
import re
import shutil
import string
import html

# Define the directory
save_dir = "../../data/raw_lapresse_dataset/"

# Clear the folder before saving new files
if os.path.exists(save_dir):
    shutil.rmtree(save_dir)  # Remove all existing files and subdirectories
os.makedirs(save_dir, exist_ok=True)  # Recreate the empty directory

# Function to clean filenames
def clean_filename(title):
    title = title.strip()  # Remove leading/trailing spaces
    title = re.sub(r'[\/:*?"<>|]', '', title)  # Replace invalid characters
    title = re.sub(r'\s+', ' ', title)  # Replace multiple spaces with a single space
    return title[:150].strip()  # Ensure no leading spaces after replacements

# Iterate over each row and save the summary as a .txt file
for _, row in df_top6000.iterrows():
    title = row['title']  # Keep title raw
    summary = row['summary']  # Keep summary raw

    # Fix encoding issues
    summary = ''.join(c for c in summary if c in string.printable)  # Remove invisible characters
    summary = summary.replace("\r\n", "\n").replace("\r", "\n")  # Normalize line endings
    summary = html.unescape(summary)  # Convert HTML entities (e.g., &#13; -> newline)

    # Create a valid filename
    filename = f"{clean_filename(title)}.txt"
    file_path = os.path.join(save_dir, filename)
    
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(summary)

print(f"Cleared the folder and saved {len(df_top6000)} files in {save_dir}")

Cleared the folder and saved 6000 files in ../datasets/dataset_lapresse/
